In [1]:
!pip install ultralytics

import os
import json
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from ultralytics import YOLO
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_DIR     = "/kaggle/input/datasets/kavishgrover/bdd-oia/data"
OUTPUT_DIR    = "/kaggle/working/features"
YOLO_WEIGHTS  = "/kaggle/working/yolov8m.pt"
MAX_NEW_FILES = 3000
device        = "cuda" if torch.cuda.is_available() else "cpu"

new_files_this_run = 0
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load already-processed filenames from previous runs ───────────────────────
# Always attach only the LATEST version of bdd-oia-processed-log as input
# It already contains full history of all previous runs
PROCESSED_LOG = "/kaggle/input/datasets/kavishgrover/features-v1/processed.txt"
already_processed = set()

if os.path.exists(PROCESSED_LOG):
    with open(PROCESSED_LOG) as f:
        already_processed = set(line.strip() for line in f)
    print(f"Loaded {len(already_processed)} already-processed filenames from log ✓")
else:
    print("No previous log found — fresh start ✓")

# Cumulative log: starts with full history, appends new entries this run
log_out = open("/kaggle/working/processed.txt", "w")
for fname in already_processed:
    log_out.write(fname + "\n")

# ── Sanity check: confirm we're starting from the right place ─────────────────
print(f"\n── Pre-run state ────────────────────────────────")
print(f"  Already processed (from log) : {len(already_processed)}")
print(f"  New files to process this run: up to {MAX_NEW_FILES}")
print(f"  Will save features to        : {OUTPUT_DIR}")
print(f"─────────────────────────────────────────────────\n")

# ── Download YOLO weights once ────────────────────────────────────────────────
if not os.path.exists(YOLO_WEIGHTS):
    print("Downloading yolov8m.pt...")
    _tmp = YOLO("yolov8m.pt")
    del _tmp
    print("Downloaded ✓")
else:
    print("yolov8m.pt already exists ✓")

yolo_model = YOLO(YOLO_WEIGHTS)

# ── ResNet-50 backbone (cut at layer4) ────────────────────────────────────────
resnet_full     = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet_backbone = nn.Sequential(
    resnet_full.conv1,
    resnet_full.bn1,
    resnet_full.relu,
    resnet_full.maxpool,
    resnet_full.layer1,
    resnet_full.layer2,
    resnet_full.layer3,
    resnet_full.layer4,
).eval().to(device)

for param in resnet_backbone.parameters():
    param.requires_grad = False

print(f"Device: {device} | Models loaded and frozen ✓")

# ── Transform ─────────────────────────────────────────────────────────────────
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# ── Feature extraction ────────────────────────────────────────────────────────
def extract_features(image_path, conf_threshold=0.3):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    H, W    = img_rgb.shape[:2]

    results   = yolo_model(image_path, conf=conf_threshold, verbose=False)
    raw_boxes = results[0].boxes.xyxy.cpu().numpy()

    valid_crops = []
    valid_boxes = []
    for box in raw_boxes:
        x_min, y_min, x_max, y_max = map(int, box)
        x_min = max(0, x_min);  y_min = max(0, y_min)
        x_max = min(W, x_max);  y_max = min(H, y_max)
        if x_max <= x_min or y_max <= y_min:
            continue
        crop = pil_img.crop((x_min, y_min, x_max, y_max))
        valid_crops.append(transform(crop))
        valid_boxes.append([x_min, y_min, x_max, y_max])

    used_fallback = False
    if len(valid_crops) == 0:
        valid_crops = [transform(pil_img)]
        valid_boxes = [[0, 0, W, H]]
        used_fallback = True

    crop_batch = torch.stack(valid_crops).to(device)
    with torch.no_grad():
        local_feats = resnet_backbone(crop_batch)        # (N, 2048, 7, 7)

    global_tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        global_map = resnet_backbone(global_tensor)      # (1, 2048, 7, 7)

    boxes = np.array(valid_boxes, dtype=np.float32)
    return local_feats.cpu(), global_map.cpu(), boxes, used_fallback

# ── Preprocessing loop ────────────────────────────────────────────────────────
def preprocess_split(split):
    global yolo_model, new_files_this_run

    actions_path = f"/kaggle/input/datasets/kavishgrover/bdd-oia/{split}_25k_images_actions.json"
    with open(actions_path) as f:
        actions_data = json.load(f)

    id_to_filename = {img["id"]: img["file_name"] for img in actions_data["images"]}

    filenames = []
    for ann in actions_data["annotations"]:
        if len(ann["category"]) == 5 and ann["category"][4] == 1:
            continue
        filenames.append(id_to_filename[ann["id"]])

    # Sanity check: how many in this split are already done vs remaining
    already_done_this_split = sum(1 for f in filenames if f in already_processed)
    remaining = len(filenames) - already_done_this_split
    print(f"\n── {split} split ────────────────────────────────────")
    print(f"  Total valid files     : {len(filenames)}")
    print(f"  Already processed     : {already_done_this_split}")
    print(f"  Remaining to process  : {remaining}")
    print(f"  Will process this run : {min(remaining, MAX_NEW_FILES - new_files_this_run)}")
    print(f"─────────────────────────────────────────────────")

    if remaining == 0:
        print(f"  {split} fully complete, skipping.")
        return

    out_dir = os.path.join(OUTPUT_DIR, split)
    os.makedirs(out_dir, exist_ok=True)

    failed        = []
    fallbacks     = []
    first_new     = None   # track first new file processed for sanity check

    for fname in tqdm(filenames, desc=f"{split}"):

        if new_files_this_run >= MAX_NEW_FILES:
            print(f"  Hit {MAX_NEW_FILES} file limit, stopping.")
            break

        if fname in already_processed:
            continue

        img_path  = os.path.join(IMAGE_DIR, fname)
        save_path = os.path.join(out_dir, fname.replace(".jpg", ".pt"))

        if not os.path.exists(img_path):
            failed.append(fname)
            continue

        try:
            local_feats, global_map, boxes, used_fallback = extract_features(img_path)

            torch.save({
                "filename"   : fname,
                "local_feats": local_feats,
                "global_map" : global_map,
                "boxes"      : torch.tensor(boxes),
            }, save_path)

            new_files_this_run += 1
            already_processed.add(fname)
            log_out.write(fname + "\n")
            log_out.flush()

            # Sanity check on the very first new file processed
            if first_new is None:
                first_new = fname
                verify = torch.load(save_path)
                print(f"\n  ✓ First new file processed: {fname}")
                print(f"    local_feats : {verify['local_feats'].shape}")
                print(f"    global_map  : {verify['global_map'].shape}")
                print(f"    boxes       : {verify['boxes'].shape}")

            if used_fallback:
                fallbacks.append(fname)

        except Exception as e:
            print(f"  Failed {fname}: {e}")
            failed.append(fname)
            if "inline_container" in str(e) or "unexpected pos" in str(e):
                print("  Reloading YOLO weights...")
                if os.path.exists(YOLO_WEIGHTS):
                    os.remove(YOLO_WEIGHTS)
                yolo_model = YOLO(YOLO_WEIGHTS)

    print(f"\n  {split} run complete.")
    print(f"    New files this run : {new_files_this_run}")
    print(f"    Total processed    : {len(already_processed)}")
    print(f"    Failed             : {len(failed)}")
    print(f"    Fallbacks          : {len(fallbacks)}")

for split in ["train", "val", "test"]:
    preprocess_split(split)

log_out.close()
print(f"\n── All done ─────────────────────────────────────")
print(f"  Total new files this run : {new_files_this_run}")
print(f"  Total processed overall  : {len(already_processed)}")
print(f"  Log saved to             : /kaggle/working/processed.txt")
print(f"─────────────────────────────────────────────────")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loaded 3000 already-processed filenames from log ✓

── Pre-run state ────────────────────────────────
  Already processed (from log) : 3000
  New files to process this run: up to 3000
  Will save features to        : /kaggle/working/features
─────────────────────────────────────────────────

Downloaded ✓
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s]


Device: cuda | Models loaded and frozen ✓

── train split ────────────────────────────────────
  Total valid files     : 16028
  Already processed     : 3000
  Remaining to process  : 13028
  Will process this run : 3000
─────────────────────────────────────────────────


train:  19%|█▊        | 3001/16028 [00:01<00:06, 1885.96it/s]


  ✓ First new file processed: 7845b80e-86451b15_1.jpg
    local_feats : torch.Size([15, 2048, 7, 7])
    global_map  : torch.Size([1, 2048, 7, 7])
    boxes       : torch.Size([15, 4])


...

  Hit 3000 file limit, stopping.

  train run complete.
    New files this run : 3000
    Total processed    : 6000
    Failed             : 0
    Fallbacks          : 7

── val split ────────────────────────────────────
  Total valid files     : 2259
  Already processed     : 0
  Remaining to process  : 2259
  Will process this run : 0
─────────────────────────────────────────────────


val:   0%|          | 0/2259 [00:00<?, ?it/s]


  Hit 3000 file limit, stopping.

  val run complete.
    New files this run : 3000
    Total processed    : 6000
    Failed             : 0
    Fallbacks          : 0

── test split ────────────────────────────────────
  Total valid files     : 4548
  Already processed     : 0
  Remaining to process  : 4548
  Will process this run : 0
─────────────────────────────────────────────────


test:   0%|          | 0/4548 [00:00<?, ?it/s]

  Hit 3000 file limit, stopping.

  test run complete.
    New files this run : 3000
    Total processed    : 6000
    Failed             : 0
    Fallbacks          : 0

── All done ─────────────────────────────────────
  Total new files this run : 3000
  Total processed overall  : 6000
  Log saved to             : /kaggle/working/processed.txt
─────────────────────────────────────────────────
